# 🤝 Experiment: Multi-Agent Critique
**Workflows Over Weights: Agentic Scaling (Qwen-7B Answerer + Llama-8B Critic)**

This notebook tests the hypothesis that a larger/different model (Llama-3.1-8B via Groq) performing the critique stage yields better results than self-critique.


---
## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
%%capture
# Install dependencies (cached after first run)
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    bitsandbytes>=0.43.0 \
    torch \
    tavily-python \
    duckduckgo-search \
    pydantic \
    python-dotenv groq

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/workflows-over-weights'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

In [ ]:
# Groq Setup for Multi-Agent Critique
from google.colab import userdata
try:
    import groq
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    groq_client = groq.Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized.")
except Exception as e:
    print(f"Groq setup failed: {e}")
    groq_client = None

In [ ]:
# Clone the repo (or pull latest)
REPO_URL = 'https://github.com/ajanm007/workflows-over-weights.git'  # UPDATE with your repo URL
REPO_DIR = '/content/workflows-over-weights'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# Add repo root to Python path
import sys
sys.path.insert(0, REPO_DIR)
print(f'Repo directory: {REPO_DIR}')
print(f'sys.path includes repo: {REPO_DIR in sys.path}')

---
## 2. Load Model (Qwen2.5-7B-Instruct, 4-bit NF4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

print(f'Loading {MODEL_ID} with 4-bit NF4 quantization...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
print(f'Model loaded. Device map: {model.hf_device_map}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

In [ ]:
# Quick sanity check — does the model generate?
test_prompt = 'What is 2 + 2? Answer with just the number.'
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=32, do_sample=False)
response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Sanity check response: "{response.strip()}"')
assert '4' in response, f'Model sanity check failed! Got: {response}'

---
## 3. LLM Generation Function
Replaces the mock_llm with actual Qwen inference.

In [ ]:
import re
from config import MAX_TOKENS

def generate_response(prompt: str, max_new_tokens: int = MAX_TOKENS, temperature: float = 0.0) -> str:
    """Generate a response from the Qwen model."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

def extract_search_query(text: str) -> str | None:
    match = re.search(r'<SEARCH>(.*?)</SEARCH>', text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

def extract_final_answer(text: str) -> str | None:
    match = re.search(r'<FINAL_ANSWER>(.*?)</FINAL_ANSWER>', text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

# Quick test
test = generate_response('What is the capital of France? Answer with <FINAL_ANSWER>your answer</FINAL_ANSWER>')
print(f'Test response: {test}')
print(f'Extracted answer: {extract_final_answer(test)}')

---
## 4. Load Dataset & Pipeline Components

In [ ]:
import json
import logging
from config import DATASET_PATH
from pipeline.prompts import BASE_PROMPT, TOOL_AUGMENTED_PROMPT, SELF_CRITIQUE_PROMPT_SYSTEM, SELF_CRITIQUE_PROMPT_USER, MULTI_AGENT_CRITIC_PROMPT_SYSTEM, MULTI_AGENT_CRITIC_PROMPT_USER, MULTI_AGENT_REWRITE_PROMPT
from tools.web_search import web_search
from eval.scorer import score_exact_match

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Load dataset
# Option A: from cloned repo
if DATASET_PATH.exists():
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    print(f'Loaded {len(dataset)} items from repo dataset.')
# Option B: from Drive (if dataset was uploaded there)
else:
    drive_dataset = os.path.join(DRIVE_DIR, 'hle_verified_gold.json')
    if os.path.exists(drive_dataset):
        with open(drive_dataset, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
        print(f'Loaded {len(dataset)} items from Drive.')
    else:
        raise FileNotFoundError('Dataset not found! Upload hle_verified_gold.json to Drive or include in repo.')

print(f'First item keys: {list(dataset[0].keys())}')
print(f'Sample question (truncated): {dataset[0]["question"][:150]}...')

---
## 5. Stage Runners (Real LLM)

In [ ]:
def run_stage_1(item):
    """Stage 1: Zero-shot baseline."""
    prompt = BASE_PROMPT.format(question=item['question'])
    response = generate_response(prompt)
    prediction = extract_final_answer(response) or response
    return {'prompt': prompt, 'response': response, 'prediction': prediction}

def run_stage_2(item):
    """Stage 2: Tool-augmented (single web search)."""
    prompt = TOOL_AUGMENTED_PROMPT.format(question=item['question'])
    response = generate_response(prompt)
    
    query = extract_search_query(response)
    search_provider = None
    if query:
        search_results = web_search(query)
        search_provider = 'tavily' if 'Tavily' in search_results else 'duckduckgo'
        prompt += f'\n\nSearch results for "{query}":\n{search_results}\n\nBased on the search results, provide your final answer with <FINAL_ANSWER>your answer</FINAL_ANSWER>.'
        response = generate_response(prompt)
    
    prediction = extract_final_answer(response) or response
    return {'prompt': prompt, 'response': response, 'prediction': prediction, 'search_query': query, 'search_provider': search_provider}

def run_stage_3(item, s1_response):
    """Stage 3: Self-critique (no tools)."""
    sys_prompt = SELF_CRITIQUE_PROMPT_SYSTEM
    user_prompt = SELF_CRITIQUE_PROMPT_USER.format(question=item['question'], candidate_response=s1_response)
    combined = f'{sys_prompt}\n{user_prompt}'
    
    response = generate_response(combined)
    prediction = extract_final_answer(response) or response
    return {'prompt': combined, 'response': response, 'prediction': prediction}

def run_stage_4(item, s2_response):
    """Stage 4: Full lean pipeline (tools + critique)."""
    # Strip nested FINAL_ANSWER tags from Stage 2 response to avoid parse confusion\n    s2_response = re.sub(r'<FINAL_ANSWER>.*?</FINAL_ANSWER>', '', s2_response, flags=re.DOTALL).strip()\n    sys_prompt = SELF_CRITIQUE_PROMPT_SYSTEM
    user_prompt = SELF_CRITIQUE_PROMPT_USER.format(question=item['question'], candidate_response=s2_response)
    combined = f'{sys_prompt}\n{user_prompt}'
    
    response = generate_response(combined)
    prediction = extract_final_answer(response) or response
    return {'prompt': combined, 'response': response, 'prediction': prediction}

def run_stage_5(item, s2_response):
    """Stage 5: Multi-Agent Critique (Llama-3.1-8B via Groq)."""
    # Strip nested FINAL_ANSWER tags from Stage 2 response\n    s2_response = re.sub(r'<FINAL_ANSWER>.*?</FINAL_ANSWER>', '', s2_response, flags=re.DOTALL).strip()\n    if not groq_client:
        return {"prediction": "SKIPPED (No Groq Key)", "response": "", "critique": ""}
    
    critique_user = MULTI_AGENT_CRITIC_PROMPT_USER.format(question=item["question"], candidate_response=s2_response)
    try:
        completion = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": MULTI_AGENT_CRITIC_PROMPT_SYSTEM},
                {"role": "user", "content": critique_user}
            ],
            temperature=0.0,
            max_tokens=1024
        )
        critique = completion.choices[0].message.content
    except Exception as e:
        return {"prediction": f"ERROR: {e}", "response": "", "critique": str(e)}
    
    rewrite_prompt = MULTI_AGENT_REWRITE_PROMPT.format(question=item["question"], original_answer=s2_response, critique=critique)
    response = generate_response(rewrite_prompt)
    prediction = extract_final_answer(response) or response
    return {"prompt": rewrite_prompt, "response": response, "prediction": prediction, "critique": critique}

---
## 6. Smoke Test — Run 10 Questions

In [ ]:
import time

SMOKE_CHECKPOINT = os.path.join(DRIVE_DIR, 'smoke_test_results.json')
SMOKE_LIMIT = 10

# Load existing checkpoint if resuming
if os.path.exists(SMOKE_CHECKPOINT):
    with open(SMOKE_CHECKPOINT, 'r') as f:
        results = json.load(f)
    print(f'Resuming from checkpoint: {len(results)} items already done.')
else:
    results = {}

for idx, item in enumerate(dataset[:SMOKE_LIMIT]):
    qid = str(item.get('id', idx))
    
    if qid in results:
        print(f'[{idx+1}/{SMOKE_LIMIT}] Skipping QID {qid} (already done)')
        continue
    
    print(f'\n{"="*60}')
    print(f'[{idx+1}/{SMOKE_LIMIT}] Processing QID: {qid}')
    print(f'Question: {item["question"][:120]}...')
    
    expected_ans = item.get('answer', '')
    ans_type = item.get('answer_type', 'exactMatch')
    
    item_results = {'expected': expected_ans, 'type': ans_type, 'stages': {}}
    
    t0 = time.time()
    
    # Stage 1
    print('  Stage 1 (baseline)...', end=' ')
    s1 = run_stage_1(item)
    s1['correct'] = score_exact_match(s1['prediction'], expected_ans, ans_type)
    item_results['stages']['1'] = s1
    print(f'Prediction: "{s1["prediction"]}" | Correct: {s1["correct"]}')
    
    # Stage 2
    print('  Stage 2 (+tools)...', end=' ')
    s2 = run_stage_2(item)
    s2['correct'] = score_exact_match(s2['prediction'], expected_ans, ans_type)
    item_results['stages']['2'] = s2
    print(f'Search: "{s2.get("search_query", "none")}" | Prediction: "{s2["prediction"]}" | Correct: {s2["correct"]}')
    
    # Stage 3
    print('  Stage 3 (+critique)...', end=' ')
    s3 = run_stage_3(item, s1['response'])
    s3['correct'] = score_exact_match(s3['prediction'], expected_ans, ans_type)
    item_results['stages']['3'] = s3
    print(f'Prediction: "{s3["prediction"]}" | Correct: {s3["correct"]}')
    
    # Stage 4
    print('  Stage 4 (full)...', end=' ')
    s4 = run_stage_4(item, s2['response'])
    s4['correct'] = score_exact_match(s4['prediction'], expected_ans, ans_type)
    item_results['stages']['4'] = s4
    print(f'Prediction: "{s4["prediction"]}" | Correct: {s4["correct"]}')
    # Stage 5 (Multi-Agent)
    print("  Stage 5 (multi-agent)...", end=" ")
    s5 = run_stage_5(item, s2["response"])
    s5["correct"] = score_exact_match(s5["prediction"], expected_ans, ans_type)
    item_results["stages"]["5"] = s5
    print(f"Prediction: '{s5['prediction']}' | Correct: {s5['correct']}")
    
    elapsed = time.time() - t0
    item_results['time_seconds'] = round(elapsed, 1)
    print(f'  Time: {elapsed:.1f}s')
    
    results[qid] = item_results
    
    # Save after every question
    with open(SMOKE_CHECKPOINT, 'w') as f:
        json.dump(results, f, indent=2)

print(f'\n{"="*60}')
print(f'Smoke test complete. Results saved to {SMOKE_CHECKPOINT}')

---
## 7. Smoke Test Summary

In [ ]:
# Compute summary statistics
stage_correct = {s: 0 for s in ['1', '2', '3', '4', '5']}
stage_parsed = {s: 0 for s in ['1', '2', '3', '4', '5']}  # Had valid <FINAL_ANSWER>
total = len(results)
total_time = 0

for qid, r in results.items():
    total_time += r.get('time_seconds', 0)
    for s in ['1', '2', '3', '4', '5']:
        if r['stages'][s]['correct']:
            stage_correct[s] += 1
        # Check if FINAL_ANSWER was successfully extracted
        if r['stages'][s]['prediction'] != r['stages'][s]['response']:
            stage_parsed[s] += 1

print('='*60)
print('SMOKE TEST SUMMARY')
print('='*60)
print(f'Total questions: {total}')
print(f'Total time: {total_time:.0f}s ({total_time/total:.1f}s per question)')
print(f'Avg per question (all 4 stages): {total_time/total:.1f}s')
print(f'Estimated full run time: {total_time/total * 641 / 60:.0f} minutes')
print()
print(f'{"Stage":<20} {"Accuracy":>10} {"FINAL_ANSWER Parsed":>22}')
print('-'*55)
for s, name in [('1', 'Baseline'), ('2', '+Tools'), ('3', '+Critique'), ('4', 'Full'), ('5', 'Multi-Agent')]:
    acc = stage_correct[s] / total * 100 if total > 0 else 0
    parsed = stage_parsed[s] / total * 100 if total > 0 else 0
    print(f'Stage {s}: {name:<13} {acc:>8.1f}%  {parsed:>18.1f}%')

print()
print('GO/NO-GO CHECKS:')
all_parse_rate = sum(stage_parsed.values()) / (total * 5) * 100 if total > 0 else 0
print(f'  [{"PASS" if all_parse_rate > 50 else "FAIL"}] FINAL_ANSWER parse rate: {all_parse_rate:.0f}% (need > 50%)')
avg_time = total_time / total if total > 0 else 999
print(f'  [{"PASS" if avg_time < 300 else "FAIL"}] Avg time per question: {avg_time:.1f}s (need < 300s for session budget)')
print(f'  [{"PASS" if torch.cuda.memory_allocated()/1024**3 < 15 else "WARN"}] GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB (T4 has 16GB)')

---
## 8. Inspect Individual Results

In [ ]:
# Inspect a specific question
inspect_idx = 0  # Change this to explore different questions
qid = list(results.keys())[inspect_idx]
r = results[qid]

print(f'QID: {qid}')
print(f'Expected: {r["expected"]}')
print(f'Type: {r["type"]}')
print()
for s in ['1', '2', '3', '4', '5']:
    stage = r['stages'][s]
    print(f'--- Stage {s} ---')
    print(f'Prediction: {stage["prediction"]}')
    print(f'Correct: {stage["correct"]}')
    if s == '2' and stage.get('search_query'):
        print(f'Search: {stage["search_query"]}')
    print(f'Response (first 300 chars):')
    print(stage['response'][:300])
    print()